# Day 9 v2 — Silver: All Realtime Blob Entities (For Loop)

**Source:** `bronze-volume/realtime/<entity>/YYYY/MM/DD/HH/`  
**Sink:** `silver-volume/realtime/<entity>/` (Delta, full overwrite)

Entities: `charging_sessions` (19 cols), `maintenance_events` (14 cols)

**Fixes vs original:**
- `try_to_timestamp()` + `try_cast()` throughout — serverless ANSI-safe
- NULL guard on natural key and `updated_at` before dedup

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime"
SILVER_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime"
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze : {BRONZE_REALTIME}")
print(f"Silver : {SILVER_REALTIME}")
print(f"Run TS : {RUN_TS}")

In [ ]:
# Cell 3 — Entity config + ANSI-safe timestamp helper

def _safe_ts(col_name):
    """Try ISO string first, fall back to Unix epoch float. Never throws on serverless."""
    iso   = F.try_to_timestamp(F.col(col_name))
    epoch = F.try_to_timestamp(F.col(col_name).try_cast("double").cast("string"))
    return F.when(iso.isNotNull(), iso).otherwise(epoch).alias(col_name)

ENTITIES = [
    {
        "entity_name": "charging_sessions",
        "natural_key": "session_id",
        "schema": StructType([
            StructField("session_id",          StringType(), True),
            StructField("charger_id",          StringType(), True),
            StructField("station_id",          StringType(), True),
            StructField("vehicle_id",          StringType(), True),
            StructField("customer_id",         StringType(), True),
            StructField("plug_in_ts",          StringType(), True),
            StructField("charge_end_ts",       StringType(), True),
            StructField("duration_min",        StringType(), True),
            StructField("energy_kwh",          StringType(), True),
            StructField("peak_kw",             StringType(), True),
            StructField("connector_type",      StringType(), True),
            StructField("session_status",      StringType(), True),
            StructField("tariff_id",           StringType(), True),
            StructField("raw_device_temp_c",   StringType(), True),
            StructField("signal_strength_dbm", StringType(), True),
            StructField("firmware_ver",        StringType(), True),
            StructField("state_code",          StringType(), True),
            StructField("protocol_version",    StringType(), True),
            StructField("ingested_at",         StringType(), True),
        ]),
        "cast_select": lambda df: df.select(
            F.col("session_id").cast("string"),
            F.col("charger_id").cast("string"),
            F.col("station_id").cast("string"),
            F.col("vehicle_id").cast("string"),
            F.col("customer_id").cast("string"),
            _safe_ts("plug_in_ts"),
            _safe_ts("charge_end_ts"),
            F.col("duration_min").try_cast("integer"),
            F.col("energy_kwh").try_cast("decimal(10,4)"),
            F.col("peak_kw").try_cast("decimal(10,4)"),
            F.col("connector_type").cast("string"),
            F.col("session_status").cast("string"),
            F.col("tariff_id").cast("string"),
            F.col("raw_device_temp_c").try_cast("decimal(6,2)"),
            F.col("signal_strength_dbm").try_cast("integer"),
            F.col("firmware_ver").cast("string"),
            F.col("state_code").cast("string"),
            F.col("protocol_version").cast("string"),
            _safe_ts("ingested_at"),
            F.col("updated_at"),
        ),
    },
    {
        "entity_name": "maintenance_events",
        "natural_key": "event_id",
        "schema": StructType([
            StructField("event_id",           StringType(), True),
            StructField("charger_id",         StringType(), True),
            StructField("station_id",         StringType(), True),
            StructField("employee_id",        StringType(), True),
            StructField("root_cause",         StringType(), True),
            StructField("mttr_hours",         StringType(), True),
            StructField("event_ts",           StringType(), True),
            StructField("device_error_code",  StringType(), True),
            StructField("voltage_at_fault",   StringType(), True),
            StructField("ambient_temp_c",     StringType(), True),
            StructField("state_code",         StringType(), True),
            StructField("part_replaced",      StringType(), True),
            StructField("raw_payload_bytes",  StringType(), True),
            StructField("ingested_at",        StringType(), True),
        ]),
        "cast_select": lambda df: df.select(
            F.col("event_id").cast("string"),
            F.col("charger_id").cast("string"),
            F.col("station_id").cast("string"),
            F.col("employee_id").cast("string"),
            F.col("root_cause").cast("string"),
            F.col("mttr_hours").try_cast("decimal(8,2)"),
            _safe_ts("event_ts"),
            F.col("device_error_code").cast("string"),
            F.col("voltage_at_fault").try_cast("decimal(8,2)"),
            F.col("ambient_temp_c").try_cast("decimal(6,2)"),
            F.col("state_code").cast("string"),
            F.col("part_replaced").cast("string"),
            F.col("raw_payload_bytes").try_cast("long"),
            _safe_ts("ingested_at"),
            F.col("updated_at"),
        ),
    },
]

print(f"Entities: {[e['entity_name'] for e in ENTITIES]}")

In [ ]:
# Cell 4 — For loop: transform all realtime blob entities
results = []

for entity in ENTITIES:
    name        = entity["entity_name"]
    natural_key = entity["natural_key"]
    csv_schema  = entity["schema"]
    csv_cols    = [f.name for f in csv_schema.fields]
    bronze_path = f"{BRONZE_REALTIME}/{name}"
    silver_path = f"{SILVER_REALTIME}/{name}"

    print(f"Processing: {name} ...", end=" ")
    try:
        # Step 1: Read Bronze CSV
        raw_df = (
            spark.read
            .option("header", "true")
            .option("recursiveFileLookup", "true")
            .schema(csv_schema)
            .csv(bronze_path)
            .select(
                *[F.col(c) for c in csv_cols],
                F.col("_metadata.file_path").alias("_file_path")
            )
        )
        bronze_rows = raw_df.count()

        # Step 2: Derive updated_at from file path partition
        raw_df = (
            raw_df
            .withColumn("_year",    F.regexp_extract(F.col("_file_path"), r"/(\d{4})/", 1))
            .withColumn("_month",   F.regexp_extract(F.col("_file_path"), r"/\d{4}/(\d{2})/", 1))
            .withColumn("_day",     F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/(\d{2})/", 1))
            .withColumn("_hour",    F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
            .withColumn("_upd_str", F.concat_ws(" ",
                                        F.concat_ws("-", F.col("_year"), F.col("_month"), F.col("_day")),
                                        F.col("_hour")))
            .withColumn("updated_at", F.to_timestamp(F.col("_upd_str"), "yyyy-MM-dd HH"))
            .drop("_file_path", "_year", "_month", "_day", "_hour", "_upd_str")
        )

        # Step 3: Cast (entity-specific)
        typed_df = entity["cast_select"](raw_df)

        # Step 4: Trim string columns
        for c, t in typed_df.dtypes:
            if t == "string":
                typed_df = typed_df.withColumn(c, F.trim(F.col(c)))

        # Step 5: Add audit columns
        audited_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit("full"))
            .withColumn("silver_pipeline",    F.lit("pl_silver_blob_all_entities_v2"))
        )

        # Step 6: NULL guard on natural key and updated_at
        audited_df = audited_df.filter(
            F.col(natural_key).isNotNull() &
            (F.trim(F.col(natural_key)) != "") &
            F.col("updated_at").isNotNull()
        )

        # Step 7: Deduplicate
        window = Window.partitionBy(natural_key).orderBy(F.col("updated_at").desc())
        deduped_df = (
            audited_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        silver_rows = deduped_df.count()

        # Step 8: Write Silver Delta
        (
            deduped_df.write.format("delta")
            .mode("overwrite").option("overwriteSchema", "true")
            .save(silver_path)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"entity": name, "status": "succeeded", "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as ex:
        print(f"FAILED: {ex}")
        results.append({"entity": name, "status": "failed", "bronze_rows": 0, "silver_rows": 0, "error": str(ex)})

print("\nAll entities done.")

In [ ]:
# Cell 5 — Run summary
succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 65)
print("SILVER BLOB REALTIME v2 — RUN SUMMARY")
print("=" * 65)
print(f"  run_ts    : {RUN_TS}")
print(f"  succeeded : {len(succeeded)}  |  failed: {len(failed)}")
print()
for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['entity']:<25} bronze={r['bronze_rows']:>8}  silver={r['silver_rows']:>8}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")